CELL 1: Imports and Setup

In [1]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import pickle
import warnings

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from dataset_cifar import get_federated_dataloaders
from models_cifar import get_resnet18_classifier
from utils_cifar import calculate_entropy, compute_dynamic_mu, evaluate_resnet
from client_cifar import train_client_resnet_dp, train_client_resnet_nodp
from server_cifar import federated_averaging

In [2]:
warnings.filterwarnings('ignore')

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch Version: {torch.__version__}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch Version: 2.6.0+cu124


In [4]:
def set_seed(seed=42):
    """Locks all random number generators for exact reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [5]:
set_seed(42)

CELL 2: Data Loading & Hybrid Weight Calculation (alpha = 0.1)

In [ ]:
num_clients = 5
num_classes = 10

client_loaders, test_loader = get_federated_dataloaders(num_clients=num_clients, alpha=0.1, batch_size=64)

class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

entropies = []
org_distributions = []
sample_counts = [] 

for i, loader in enumerate(client_loaders):
    # Extract labels directly from the underlying Subset indices for speed
    subset = loader.dataset
    labels = np.array(subset.dataset.targets)[subset.indices]
    entropies.append(calculate_entropy(labels, num_classes=num_classes))
    
    counts = np.bincount(labels, minlength=num_classes)
    org_distributions.append(counts)
    sample_counts.append(len(labels)) 

# Lower lambda_scale at 0.5 to prevent paralyzing skewed organizations
dynamic_mus = compute_dynamic_mu(entropies, num_classes=num_classes, mu_base=0.01, lambda_scale=0.5)

# Pure Exponential Entropy Weighting
fedshield_weights = [np.exp(e) for e in entropies]

# Sub-linear (Square-Root) Volume + Exponential Entropy Weighting
fedshield_weights_v2 = [np.sqrt(n) * np.exp(e) for e, n in zip(entropies, sample_counts)]

# Display Formatted Matrix
df_dist = pd.DataFrame(org_distributions, columns=[f"{c} (C{idx})" for idx, c in enumerate(class_names)])
df_dist.insert(0, "Organization", [f"Org {i+1}" for i in range(num_clients)])
df_dist["Total Samples"] = sample_counts
df_dist["Entropy (H_k)"] = [f"{h:.4f}" for h in entropies]
df_dist["Assigned mu_k"] = [f"{m:.4f}" for m in dynamic_mus]
df_dist["Exp Weight"] = [f"{w:.2f}" for w in fedshield_weights] 

print("\n========================= ORGANIZATION CLASS DISTRIBUTION MATRIX =========================")
print(df_dist.to_string(index=False))
print("==========================================================================================\n")

print("--- FedShieldFull2 Hybrid Weights ---")
for i in range(num_clients):
    print(f"Org {i+1} | sqrt(n) * exp(H): {fedshield_weights_v2[i]:.4f}")

CELL 3: DP Training Function

In [ ]:
def run_resnet_experiment(algorithm_name="FedShield", num_rounds=15):
    print(f"\n========== STARTING DP PHASE: {algorithm_name} ==========")
    
    global_model = get_resnet18_classifier(num_classes=10, freeze_backbone=True).to(device)
    auc_history = []
    final_eps = 0.0
    final_targets, final_preds = [], []

    for round_num in range(num_rounds):
        client_models = []
        is_final_round = (round_num == num_rounds - 1)
        
        for i in range(num_clients):
            local_model = copy.deepcopy(global_model)
            
            if algorithm_name == "FedAvg":
                current_mu = 0.0
            elif algorithm_name == "FedProx":
                current_mu = 0.01
            else:
                current_mu = dynamic_mus[i]
                
            trained_model, eps = train_client_resnet_dp(
                local_model=local_model, global_model=global_model,
                train_loader=client_loaders[i], mu=current_mu,
                device=device, epochs=2, lr=0.1, 
                noise_multiplier=1.2, max_grad_norm=1.0
            )
            client_models.append(trained_model)
            final_eps = eps

        # Server Aggregation Logic
        if algorithm_name == "FedShield":
            global_model = federated_averaging(global_model, client_models, weights=fedshield_weights)
        elif algorithm_name == "FedShieldFull2":
            global_model = federated_averaging(global_model, client_models, weights=fedshield_weights_v2)
        else: # FedAvg, FedProx, FedShield_Local all route here for n_k / N
            global_model = federated_averaging(global_model, client_models, weights=sample_counts) 
        
        # Grab predictions only on the final round
        if is_final_round:
            acc, auc, all_targets, all_preds = evaluate_resnet(global_model, test_loader, device, return_preds=True)
            final_targets, final_preds = all_targets, all_preds
        else:
            acc, auc = evaluate_resnet(global_model, test_loader, device, return_preds=False)
            
        auc_history.append(auc)
        print(f"Round {round_num + 1:02d}/{num_rounds:02d} | Global Test AUC: {auc:.4f} | Accuracy: {acc:.4f}")

    print(f"\nFinal {algorithm_name} AUC: {auc_history[-1]:.4f} | Epsilon: {final_eps:.2f}")
    
    # Print full 10-class report and plot the 10x10 matrix
    print(f"\n--- {algorithm_name} (DP) Classification Report ---")
    print(classification_report(final_targets, final_preds, target_names=class_names, digits=4, zero_division=0))
    
    cm = confusion_matrix(final_targets, final_preds)
    fig, ax = plt.subplots(figsize=(9, 9))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues, values_format='d', ax=ax)
    plt.title(f"{algorithm_name} DP Confusion Matrix (CIFAR-10)")
    plt.xticks(rotation=45) 
    plt.tight_layout()
    plt.show()
    
    return auc_history, global_model

Cell 4: Run DP Experiments & Plot

In [ ]:
auc_fedavg, _ = run_resnet_experiment("FedAvg", num_rounds=15)


In [ ]:
auc_fedprox, _ = run_resnet_experiment("FedProx", num_rounds=15)


In [ ]:

auc_fedshield, final_model_1 = run_resnet_experiment("FedShield", num_rounds=15)


In [ ]:

auc_fedshield2, final_model_2 = run_resnet_experiment("FedShieldFull2", num_rounds=15)

In [ ]:
plt.figure(figsize=(10, 6))
rounds_range = range(1, 16)

plt.plot(rounds_range, auc_fedavg, marker='o', linestyle='--', color='tab:red', label='FedAvg (mu = 0.0)')
plt.plot(rounds_range, auc_fedprox, marker='s', linestyle='-.', color='tab:orange', label='FedProx (Static mu = 0.01)')
plt.plot(rounds_range, auc_fedshield, marker='^', linestyle='-', color='tab:blue', linewidth=2.5, label='FedShield (Dynamic Shannon Entropy mu_k)')
plt.plot(rounds_range, auc_fedshield2, marker='d', linestyle=':', color='tab:green', linewidth=2.5, label='FedShieldFull2 (sqrt(n)*exp(H_k) + mu_k)')

plt.title("DP Regime: Multi-Class Image Classification (CIFAR-10)", fontsize=13, fontweight='bold')
plt.xlabel("Communication Rounds", fontsize=11)
plt.ylabel("Global Multi-Class ROC-AUC (One-vs-One)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
torch.save(final_model_1.state_dict(), "fedshield_resnet18_cifar10_augmented_v1.pth")
torch.save(final_model_2.state_dict(), "2_fedshield_resnet18_cifar10_augmented_v1.pth")

Cell 5: Non-DP Training Function (Ablation Study)

In [ ]:
def run_resnet_experiment_nodp(algorithm_name="FedShield", num_rounds=15):
    print(f"\n========== STARTING NON-DP PHASE: {algorithm_name} ==========")
    
    global_model = get_resnet18_classifier(num_classes=10, freeze_backbone=True).to(device)
    auc_history = []
    final_targets, final_preds = [], []

    for round_num in range(num_rounds):
        client_models = []
        is_final_round = (round_num == num_rounds - 1)
        
        for i in range(num_clients):
            local_model = copy.deepcopy(global_model)
            
            if algorithm_name == "FedAvg":
                current_mu = 0.0
            elif algorithm_name == "FedProx":
                current_mu = 0.01
            else:
                current_mu = dynamic_mus[i]
                
            trained_model, _ = train_client_resnet_nodp(
                local_model=local_model, global_model=global_model,
                train_loader=client_loaders[i], mu=current_mu,
                device=device, epochs=2, lr=0.1
            )
            client_models.append(trained_model)

        # Server Aggregation Logic
        if algorithm_name == "FedShield_Full":
            global_model = federated_averaging(global_model, client_models, weights=fedshield_weights)
        elif algorithm_name == "FedShieldFull2":
            global_model = federated_averaging(global_model, client_models, weights=fedshield_weights_v2)
        else: # FedAvg, FedProx, FedShield_Local all route here for n_k / N
            global_model = federated_averaging(global_model, client_models, weights=sample_counts)
        
        if is_final_round:
            acc, auc, all_targets, all_preds = evaluate_resnet(global_model, test_loader, device, return_preds=True)
            final_targets, final_preds = all_targets, all_preds
        else:
            acc, auc = evaluate_resnet(global_model, test_loader, device, return_preds=False)
            
        auc_history.append(auc)
        print(f"Round {round_num + 1:02d}/{num_rounds:02d} | Global Test AUC: {auc:.4f} | Accuracy: {acc:.4f}")

    print(f"\nFinal Non-DP {algorithm_name} AUC: {auc_history[-1]:.4f}")
    
    print(f"\n--- {algorithm_name} (No-DP) Classification Report ---")
    print(classification_report(final_targets, final_preds, target_names=class_names, digits=4, zero_division=0))
    
    cm = confusion_matrix(final_targets, final_preds)
    fig, ax = plt.subplots(figsize=(9, 9))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Greens, values_format='d', ax=ax)
    plt.title(f"{algorithm_name} No-DP Confusion Matrix (CIFAR-10)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return auc_history, global_model

CELL 6: Run Non-DP Experiments

In [ ]:
print("Starting Non-DP Ablation Study on CIFAR-10...")
nodp_auc_fedavg, nodp_model_fedavg = run_resnet_experiment_nodp("FedAvg", num_rounds=15)


In [ ]:
nodp_auc_fedprox, nodp_model_fedprox = run_resnet_experiment_nodp("FedProx", num_rounds=15)


In [ ]:
nodp_auc_fedshield_local, nodp_model_fedshield_local = run_resnet_experiment_nodp("FedShield_Local", num_rounds=15)


In [ ]:
nodp_auc_fedshield_full, nodp_model_fedshield_full = run_resnet_experiment_nodp("FedShield_Full", num_rounds=15)


In [ ]:
nodp_auc_fedshield_full_2, nodp_model_fedshield_full_2 = run_resnet_experiment_nodp("FedShieldFull2", num_rounds=15)

CELL 7: Plot Non-DP Ablation Study Results & Save Checkpoint

In [ ]:
plt.figure(figsize=(10, 6))
rounds_range = range(1, 16)

plt.plot(rounds_range, nodp_auc_fedavg, marker='o', linestyle='--', color='tab:red', label='FedAvg (No DP)')
plt.plot(rounds_range, nodp_auc_fedprox, marker='s', linestyle='-.', color='tab:orange', label='FedProx (No DP)')
plt.plot(rounds_range, nodp_auc_fedshield_local, marker='^', linestyle='-', color='tab:blue', linewidth=2, label='FedShield (Local Penalty Only)')
plt.plot(rounds_range, nodp_auc_fedshield_full, marker='D', linestyle='-', color='tab:green', linewidth=3, label='FedShield (Full: Local + exp(H_k))')
plt.plot(rounds_range, nodp_auc_fedshield_full_2, marker='d', linestyle=':', color='tab:purple', linewidth=3, label='FedShield2 (Full: Local + sqrt(n)*exp(H_k))')

plt.title("Full Ablation Study: Multi-Class Image Classification (CIFAR-10)", fontsize=13, fontweight='bold')
plt.xlabel("Communication Rounds", fontsize=11)
plt.ylabel("Global Multi-Class ROC-AUC (One-vs-One)", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Bundle EVERYTHING (DP and Non-DP histories) into one master dictionary
full_context = {
    "auc_fedshield": auc_fedshield,
    "auc_fedshield2": auc_fedshield2,
    "auc_fedprox": auc_fedprox,
    "auc_fedavg": auc_fedavg,
    "entropies": entropies,
    "dynamic_mus": dynamic_mus,
    "fedshield_weights": fedshield_weights,
    "fedshield_weights_v2": fedshield_weights_v2,
    "org_distributions": org_distributions,
    "sample_counts": sample_counts,
    "nodp_auc_fedavg": nodp_auc_fedavg,
    "nodp_auc_fedprox": nodp_auc_fedprox,
    "nodp_auc_fedshield_local": nodp_auc_fedshield_local,
    "nodp_auc_fedshield_full": nodp_auc_fedshield_full,
    "nodp_auc_fedshield_full_2": nodp_auc_fedshield_full_2
}

with open("full_cifar10_experiment_checkpoint_v1.pkl", "wb") as f:
    pickle.dump(full_context, f)

torch.save(final_model_2.state_dict(), "fedshield_dp_cifar10_model_v1.pth")
torch.save(nodp_model_fedavg.state_dict(), "nodp_fedavg_cifar10_model_v1.pth")
torch.save(nodp_model_fedprox.state_dict(), "nodp_fedprox_cifar10_model_v1.pth")
torch.save(nodp_model_fedshield_local.state_dict(), "nodp_fedshield_local_cifar10_model_v1.pth")
torch.save(nodp_model_fedshield_full.state_dict(), "nodp_fedshield_full_cifar10_model_v1.pth")
torch.save(nodp_model_fedshield_full_2.state_dict(), "nodp_fedshield_full_2_cifar10_model_v1.pth")

print("✅ ENTIRE CIFAR-10 kernel state and all models successfully saved!")